Purpose
---
This script consolidates national vessel registry data and EU Fleet Register (EUFR) data
into a unified normalized CSV dataset. It harmonizes heterogeneous schemas, merges
equivalent fields, cleans and standardizes textual values, resolves country-specific
mappings, and enriches vessel records with consolidated operational, technical, and
registry information.

# Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd "/content/drive/MyDrive/BCKG/CSV Barcos"

In [ ]:
COUNTRY = "ESP"
INPUT_MODE = "national_eu"   # options: "national_eu", "eu_only"
NATIONAL_PATH = f"national_data_by_country/national_{COUNTRY}_data.csv"
EU_PATH = f"eufr_data_by_country/eufr_data_{COUNTRY}.csv"
OUTPUT_PATH = f"consolidated_data_by_country/{INPUT_MODE}/consolidated_data_{COUNTRY}.csv"

In [ ]:
import pandas as pd
import numpy as np
import re

materiales_eu = {
    1: "Wood",
    2: "Metal",
    3: "Fiber/Plastic",
    4: "Other",
    5: "Unknown",
    6: "Polyester",
    7: "Aluminium"
}

eventos_eu = {
    "CEN": "Census",
    "CHA": "Change of Activity",
    "CST": "New Construction",
    "DES": "Destruction, wreck",
    "EXP": "Exportation, transfer",
    "IMP": "Importation, transfer",
    "MOD": "Modification",
    "RET": "Change of activity (exit)"
}

# Utilities

In [ ]:
"""Normalize CSV cell values and convert empty strings to None."""
def clean_value(v):
    if pd.isna(v):
        return None
    v = str(v).strip()
    return v if v != "" else None

"""Merge two values preserving uniqueness and returning pipe-separated output when needed."""
def merge_values(a, b):
    a = clean_value(a)
    b = clean_value(b)

    if a is None and b is None:
        return np.nan
    if a is None:
        return b
    if b is None:
        return a
    if a == b:
        return a

    values = []
    for v in [a, b]:
        if v not in values:
            values.append(v)
    return "|".join(values)

"""Merge an arbitrary number of values preserving unique non-empty entries."""
def merge_many(*values):
    cleaned = []
    for v in values:
        v = clean_value(v)
        if v is not None and v not in cleaned:
            cleaned.append(v)

    if not cleaned:
        return np.nan
    if len(cleaned) == 1:
        return cleaned[0]
    return "|".join(cleaned)

"""Return the first non-empty value from a list of candidate columns."""
def first_existing(row, *cols):
    for col in cols:
        if col in row.index:
            v = clean_value(row[col])
            if v is not None:
                return v
    return np.nan

"""Merge semantically equivalent columns into a single normalized value."""
def merge_equivalent_columns(row, *cols):
    values = []
    for col in cols:
        if col in row.index:
            v = clean_value(row[col])
            if v is not None and v not in values:
                values.append(v)

    if not values:
        return np.nan
    if len(values) == 1:
        return values[0]
    return "|".join(values)

"""Translate EU hull material numeric codes into normalized labels."""
def map_material_eu(v):
    v = clean_value(v)
    if v is None:
        return None
    try:
        return materiales_eu.get(int(v), v)
    except Exception:
        return v

"""Build a consolidated fishing gear field from multiple source columns."""
def build_fishing_gear(row):
    cols = [
        "Main fishing gear",
        "Subsidiary fishing gear 1",
        "Subsidiary fishing gear 2",
        "Subsidiary fishing gear 3",
        "Subsidiary fishing gear 4",
        "Subsidiary fishing gear 5"
    ]

    values = []
    for col in cols:
        if col in row.index:
            v = clean_value(row[col])
            if v is not None and v not in values:
                values.append(v)

    if not values:
        return np.nan
    return ", ".join(values)

"""Extract and normalize the operational census/category field."""
def extract_operates_at(v):
    v = clean_value(v)
    if v is None:
        return np.nan

    if " EN " in v:
        part = v.split(" EN ", 1)[1].strip()
        if part.startswith("EL "):
            part = part[3:].strip()
        return part if part else np.nan

    return np.nan

"""Return the event date only when the requested event code matches."""
def event_date_if(row, code):
    event = clean_value(row.get("Event"))
    date = clean_value(row.get("Event Start Date"))

    if event == code and event in eventos_eu and date is not None:
        return date
    return np.nan

"""Attempt to repair common Latin1/UTF-8 mojibake encoding issues."""
def fix_mojibake(v):
    v = clean_value(v)
    if v is None:
        return np.nan

    try:
        repaired = v.encode("latin1").decode("utf-8")
        return repaired
    except Exception:
        return v

"""Normalize Spanish registration place names for consistent matching."""
def normalize_registration_place_esp(value):
    if pd.isna(value):
        return value

    text = str(value).strip()
    if not text:
        return text

    # 1. Remove prefixes such as "34100 - "
    text = re.sub(r"^\s*\d+\s*-\s*", "", text)

    # 2. Remove any parenthesized clarification text (y el espacio previo)
    text = re.sub(r"\s*\(.*?\)", "", text)

    # 3. Keep only the first segment before commas
    text = text.split(",")[0].strip()

    # 4. Normalize repeated whitespace
    text = re.sub(r"\s+", " ", text)

    return text


In [ ]:
"""Ensure that a dataframe contains all required columns."""
def ensure_columns(df, columns, default=np.nan):
    for col in columns:
        if col not in df.columns:
            df[col] = default
    return df

"""Load, merge and normalize national/EU vessel datasets."""
def prepare_dataframe(input_mode=INPUT_MODE, nat_path=NATIONAL_PATH, eu_path=EU_PATH):
    csv2 = pd.read_csv(eu_path, sep=";", dtype=str, encoding="utf-8")
    csv2.rename(columns={"CFR": "CFR"}, inplace=True)

    if input_mode == "national_eu":
        csv1 = pd.read_csv(nat_path, sep=";", dtype=str)
        csv1.rename(columns={"CFR": "CFR"}, inplace=True)

        # Part 1: legacy behaviour -> all national registry vessels,
        # enriched with EUFR data when a CFR match exists
        df_main = csv1.merge(csv2, on="CFR", how="left", suffixes=("_csv1", "_csv2"))

        # Part 2: vessels present in EUFR but missing from the national registry
        missing_eu = csv2[~csv2["CFR"].isin(csv1["CFR"])].copy()

        # Add empty national columns to keep schema compatibility
        national_only_cols = [col for col in csv1.columns if col != "CFR"]
        for col in national_only_cols:
            if col not in missing_eu.columns:
                missing_eu[col] = np.nan

        # Reorder columns to match the merged dataframe schema
        for col in df_main.columns:
            if col not in missing_eu.columns:
                missing_eu[col] = np.nan
        missing_eu = missing_eu[df_main.columns]

        # Final union:
        # - all national vessels
        # - plus EUFR vessels missing from the national dataset
        df = pd.concat([df_main, missing_eu], ignore_index=True)

    elif input_mode == "eu_only":
        df = csv2.copy()

    else:
        raise ValueError("INPUT_MODE must be 'national_eu' o 'eu_only'")

    expected_cols = [
        "CFR",
        "MMSI",
        "Country of Registration",
        "Administración responsable del Registro",
        "Administration responsible of registration",
        "Alta en RGFP",
        "Registration on national registry",
        "Tonnage GT",
        "Other tonnage",
        "GTs",
        "Código",
        "Code",
        "LOA",
        "LBP",
        "Nombre",
        "Name",
        "Registration Number",
        "External marking",
        "Place of registration name",
        "Estado",
        "Status on national registry",
        "IRCS indicator",
        "IRCS_csv2",
        "IRCS",
        "VMS indicator",
        "ERS indicator",
        "AIS indicator",
        "UVI",
        "IMO",
        "Vessel Type",
        "Censo por modalidad",
        "Operates at",
        "Main fishing gear",
        "Subsidiary fishing gear 1",
        "Subsidiary fishing gear 2",
        "Subsidiary fishing gear 3",
        "Subsidiary fishing gear 4",
        "Subsidiary fishing gear 5",
        "Potencia",
        "Power of main engine",
        "Power of auxiliary engine",
        "Material del casco",
        "Hull material",
        "Image",
        "Date of entry into service",
        "Segment",
        "Year of construction",
        "Event",
        "Event Start Date"
    ]
    df = ensure_columns(df, expected_cols)

    df["IRCS_csv2"] = df.apply(lambda row: first_existing(row, "IRCS_csv2", "IRCS"), axis=1)

    return df

COUNTRY_CONFIG = {
    "ESP": {
            "Name of vessel": "Nombre",
            "Status on national registry": "Estado",
            "Administration responsible of registration": "Administración responsable del Registro",
            "Registration on national registry": "Alta en RGFP",
            "Code": "Código",
            "Operates at": "Censo por modalidad",
            "Power of main engine": "Potencia",
            "Hull material": "Material del casco",
            "Image": "Imagen",
            "Registration Place": "Puerto base"
    }
}

"""Resolve country-specific national column mappings."""
def resolve_national_property(prop):
    if INPUT_MODE == "national_eu":
        return COUNTRY_CONFIG[COUNTRY][prop]
    elif INPUT_MODE == "eu_only":
        return prop
    else:
        raise ValueError("INPUT_MODE must be 'national_eu' o 'eu_only'")


In [ ]:
df = prepare_dataframe()

registration_place_nat = df[resolve_national_property("Registration Place")].apply(
    normalize_registration_place_esp
)

csv3 = pd.DataFrame({
    "CFR": df["CFR"],
    "MSSI": df["MMSI"],
    "Country of registration": df["Country of Registration"],

    "Administration responsible of registration": df.apply(
        lambda row: first_existing(
            row,
            resolve_national_property("Administration responsible of registration"),
            "Administration responsible of registration"
        ),
        axis=1
    ),

    "Registration on national registry": df.apply(
        lambda row: first_existing(
            row,
            resolve_national_property("Registration on national registry"),
            "Registration on national registry"
        ),
        axis=1
    ),

    "GT Tonnage": df["Tonnage GT"],
    "Other tonnage": df["Other tonnage"],
    "GTs": df["GTs"],

    "Code": df.apply(
        lambda row: first_existing(
            row,
            resolve_national_property("Code"),
            "Code"
        ),
        axis=1
    ),

    "LOA": df["LOA"],
    "LBP": df["LBP"],

    "Name": df.apply(
        lambda row: first_existing(
            row,
            resolve_national_property("Name of vessel"),
            "Name of vessel"
        ),
        axis=1
    ),

    "Registration number": df["Registration Number"],
    "External marking": df["External marking"],
    "Registration Place": df["Place of registration name"].combine_first(registration_place_nat),
    "Status on national registry": df.apply(
        lambda row: first_existing(
            row,
            resolve_national_property("Status on national registry"),
            "Status on national registry"
        ),
        axis=1
    ),

    "Radio": df["IRCS indicator"],
    "IRCS": df["IRCS_csv2"],
    "VMS": df["VMS indicator"],
    "ERS": df["ERS indicator"],
    "AIS": df["AIS indicator"],
    "UVI": df["UVI"],
    "IMO": df["IMO"],
    "Vessel Type": df["Vessel Type"],

    "Operates at": df.apply(
        lambda row: first_existing(
            row,
            resolve_national_property("Operates at"),
            "Operates at"
        ),
        axis=1
    ).apply(extract_operates_at),

    "Fishing gear": df.apply(build_fishing_gear, axis=1),

    "Main engine": df.apply(
        lambda row: merge_values(
            row.get(resolve_national_property("Power of main engine")),
            row.get("Power of main engine")
        ),
        axis=1
    ),

    "Power of auxiliary engine": df["Power of auxiliary engine"],

    "Hull material": df.apply(
        lambda row: (
            map_material_eu(row.get("Hull material"))
            if INPUT_MODE == "eu_only"
            else merge_values(
                row.get(resolve_national_property("Hull material")),
                map_material_eu(row.get("Hull material"))
            )
        ),
        axis=1
    ),

    "Image": df.apply(
        lambda row: first_existing(
            row,
            resolve_national_property("Image"),
            "Image"
        ),
        axis=1
    ),

    "Date of entry into service": df["Date of entry into service"],
    "Segment": df["Segment"],
    "Year of construction": df["Year of construction"],
    "Year of destruction": df.apply(lambda row: event_date_if(row, "DES"), axis=1),
    "Year of retirement": df.apply(lambda row: event_date_if(row, "RET"), axis=1)
})

text_columns_to_fix = [
    "Name",
    "Registration Place"
]

for col in text_columns_to_fix:
    if col in csv3.columns:
        csv3[col] = csv3[col].apply(fix_mojibake)

if INPUT_MODE == "national_eu":
    nat = pd.read_csv(NATIONAL_PATH, sep=";", dtype=str)
    eu = pd.read_csv(EU_PATH, sep=";", dtype=str)

    nat_cfr = set(nat["CFR"].dropna().astype(str).str.strip())
    eu_cfr = set(eu["CFR"].dropna().astype(str).str.strip())
    out_cfr = set(csv3["CFR"].dropna().astype(str).str.strip())

    print("National CFR:", len(nat_cfr))
    print("EUFR CFR:", len(eu_cfr))
    print("Output CFR:", len(out_cfr))
    print("EUFR missing in national:", len(eu_cfr - nat_cfr))
    print("Still missing from output:", len((nat_cfr | eu_cfr) - out_cfr))

csv3.to_csv(OUTPUT_PATH, index=False, sep=";")
print(f"CSV generated with INPUT_MODE={INPUT_MODE}")